# Phase I — Data Preprocessing
## Goal: Clean the raw data and prepare it for feature engineering

Steps:
1. Load raw data
2. Remove invalid rows (price = 0)
3. Apply log1p transform on target
4. Handle missing values
5. Split category_name into 3 levels
6. Save cleaned data

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 60)

print("Libraries loaded ✅")

Libraries loaded ✅


In [2]:
df = pd.read_csv('../data/raw/train.tsv', sep='\t')

print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Shape: (1482535, 8)
Columns: ['train_id', 'name', 'item_condition_id', 'category_name', 'brand_name', 'price', 'shipping', 'item_description']


In [3]:
# Remove Invalid Rows

print(f"Total rows before cleaning : {len(df)}")
print(f"Rows with price == 0       : {(df['price'] == 0).sum()}")

# Remove price == 0 rows
df = df[df['price'] > 0].reset_index(drop=True)

print(f"Total rows after cleaning  : {len(df)}")
print(f"Rows removed               : {1482535 - len(df)}")


Total rows before cleaning : 1482535
Rows with price == 0       : 874
Total rows after cleaning  : 1481661
Rows removed               : 874


In [4]:
# Apply log1p on Target

# Create log-transformed target
df['log_price'] = np.log1p(df['price'])

print("Sample — original price vs log_price:")
print(df[['price', 'log_price']].head(10).to_string())

print(f"\nlog_price stats:")
print(df['log_price'].describe().round(3))

Sample — original price vs log_price:
   price  log_price
0   10.0   2.397895
1   52.0   3.970292
2   10.0   2.397895
3   35.0   3.583519
4   44.0   3.806662
5   59.0   4.094345
6   64.0   4.174387
7    6.0   1.945910
8   19.0   2.995732
9    8.0   2.197225

log_price stats:
count    1481661.000
mean           2.981
std            0.746
min            1.386
25%            2.398
50%            2.890
75%            3.401
max            7.606
Name: log_price, dtype: float64


In [5]:
# Handle Missing Values

# print("Missing values BEFORE:")
# print(df[['brand_name', 'category_name', 'item_description']].isnull().sum())

# # Fill missing values
# df['brand_name'] = df['brand_name'].fillna('missing')
# df['category_name'] = df['category_name'].fillna('missing/missing/missing')
# df['item_description'] = df['item_description'].fillna('no description')

# print("\nMissing values AFTER:")
# print(df[['brand_name', 'category_name', 'item_description']].isnull().sum())

# print("\n✅ No more missing values in these columns")


print("Missing values BEFORE:")
print(df[['brand_name', 'category_name', 'item_description']].isnull().sum())

# NEW: is_branded feature — BEFORE filling nulls
# 1 = has brand, 0 = no brand
# এটা করতে হবে fillna এর আগে, কারণ null মানেই no brand
df['is_branded'] = df['brand_name'].notna().astype(int)

# Fill missing values
df['brand_name'] = df['brand_name'].fillna('no brand')      # 'missing' → 'no brand'
df['category_name'] = df['category_name'].fillna('missing/missing/missing')
df['item_description'] = df['item_description'].fillna('no description')

print("\nMissing values AFTER:")
print(df[['brand_name', 'category_name', 'item_description']].isnull().sum())

print("\nis_branded distribution:")
print(df['is_branded'].value_counts())
print(f"\nBranded items    : {df['is_branded'].sum():,} ({df['is_branded'].mean()*100:.1f}%)")
print(f"Non-branded items: {(df['is_branded']==0).sum():,} ({(1-df['is_branded'].mean())*100:.1f}%)")

Missing values BEFORE:
brand_name          632336
category_name         6314
item_description         6
dtype: int64

Missing values AFTER:
brand_name          0
category_name       0
item_description    0
dtype: int64

is_branded distribution:
is_branded
1    849325
0    632336
Name: count, dtype: int64

Branded items    : 849,325 (57.3%)
Non-branded items: 632,336 (42.7%)


In [6]:
# Split category_name into 3 Levels

# Split category_name on '/' into 3 parts
# n=2 means split maximum 2 times → gives 3 parts
cat_split = df['category_name'].str.split('/', n=2, expand=True)

df['cat1'] = cat_split[0].fillna('missing')   # L1: Women, Electronics etc
df['cat2'] = cat_split[1].fillna('missing')   # L2: Tops, Cell Phones etc
df['cat3'] = cat_split[2].fillna('missing')   # L3: T-Shirts, iPhone etc

# Verify
print("Sample category splits:")
print(df[['category_name', 'cat1', 'cat2', 'cat3']].head(8).to_string())

print(f"\nUnique L1 categories : {df['cat1'].nunique()}")
print(f"Unique L2 categories : {df['cat2'].nunique()}")
print(f"Unique L3 categories : {df['cat3'].nunique()}")

Sample category splits:
                                        category_name               cat1                 cat2                cat3
0                                   Men/Tops/T-shirts                Men                 Tops            T-shirts
1  Electronics/Computers & Tablets/Components & Parts        Electronics  Computers & Tablets  Components & Parts
2                         Women/Tops & Blouses/Blouse              Women       Tops & Blouses              Blouse
3                  Home/Home Décor/Home Décor Accents               Home           Home Décor  Home Décor Accents
4                             Women/Jewelry/Necklaces              Women              Jewelry           Necklaces
5                                   Women/Other/Other              Women                Other               Other
6                            Women/Swimwear/Two-Piece              Women             Swimwear           Two-Piece
7                     Sports & Outdoors/Apparel/Girls  Sports & 

In [7]:
# Clean Text Columns

def clean_text(text: str) -> str:
    """
    Basic text cleaning:
    - lowercase
    - [rm] tag remove (Kaggle's price removal marker)
    - extra whitespace remove
    """
    text = str(text).lower()
    text = text.replace('[rm]', '')       # remove price-redacted markers
    text = ' '.join(text.split())         # collapse multiple spaces
    return text

# Apply to both text columns
df['name_clean'] = df['name'].apply(clean_text)
df['desc_clean'] = df['item_description'].apply(clean_text)

# Verify
print("Original name      :", df['name'].iloc[0])
print("Cleaned name       :", df['name_clean'].iloc[0])
print("\nOriginal desc      :", df['item_description'].iloc[0][:80])
print("Cleaned desc       :", df['desc_clean'].iloc[0][:80])

Original name      : MLB Cincinnati Reds T Shirt Size XL
Cleaned name       : mlb cincinnati reds t shirt size xl

Original desc      : No description yet
Cleaned desc       : no description yet


In [12]:
# # Final Shape Check

# print("="*50)
# print("PREPROCESSING SUMMARY")
# print("="*50)
# print(f"Total rows          : {len(df)}")
# print(f"Total columns       : {len(df.columns)}")
# print(f"\nColumns now:")
# for col in df.columns:
#     print(f"  {col}")

# print(f"\nMissing values (all columns):")
# missing = df.isnull().sum()
# print(missing[missing > 0] if missing.any() else "  ✅ Zero missing values")

# print(f"\nTarget column (log_price):")
# print(f"  min  : {df['log_price'].min():.3f}")
# print(f"  max  : {df['log_price'].max():.3f}")
# print(f"  mean : {df['log_price'].mean():.3f}")

print("="*50)
print("PREPROCESSING SUMMARY")
print("="*50)
print(f"Total rows          : {len(df)}")
print(f"Total columns       : {len(df.columns)}")
print(f"\nColumns now:")
for col in df.columns:
    print(f"  {col}")

print(f"\nMissing values (all columns):")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "  ✅ Zero missing values")

print(f"\nTarget column (log_price):")
print(f"  min  : {df['log_price'].min():.3f}")
print(f"  max  : {df['log_price'].max():.3f}")
print(f"  mean : {df['log_price'].mean():.3f}")

print(f"\nis_branded:")
print(f"  Branded     : {df['is_branded'].sum():,}")
print(f"  Non-branded : {(df['is_branded']==0).sum():,}")

PREPROCESSING SUMMARY
Total rows          : 1481661
Total columns       : 15

Columns now:
  train_id
  name
  item_condition_id
  category_name
  brand_name
  price
  shipping
  item_description
  log_price
  is_branded
  cat1
  cat2
  cat3
  name_clean
  desc_clean

Missing values (all columns):
  ✅ Zero missing values

Target column (log_price):
  min  : 1.386
  max  : 7.606
  mean : 2.981

is_branded:
  Branded     : 849,325
  Non-branded : 632,336


In [13]:
# import os

# # Create processed folder if not exists
# os.makedirs('../data/processed', exist_ok=True)

# # Save as pickle (fast, preserves dtypes)
# df.to_pickle('../data/processed/train_cleaned.pkl')

# # Verify save
# df_check = pd.read_pickle('../data/processed/train_cleaned.pkl')
# print(f"Saved and reloaded successfully ✅")
# print(f"Shape: {df_check.shape}")
# print(f"File size: {os.path.getsize('../data/processed/train_cleaned.pkl') / 1e6:.1f} MB")


import os
os.makedirs('../data/processed', exist_ok=True)

df.to_pickle('../data/processed/train_cleaned.pkl')

df_check = pd.read_pickle('../data/processed/train_cleaned.pkl')
print(f"Saved successfully ✅")
print(f"Shape: {df_check.shape}")
print(f"Columns: {list(df_check.columns)}")

Saved successfully ✅
Shape: (1481661, 15)
Columns: ['train_id', 'name', 'item_condition_id', 'category_name', 'brand_name', 'price', 'shipping', 'item_description', 'log_price', 'is_branded', 'cat1', 'cat2', 'cat3', 'name_clean', 'desc_clean']


## Preprocessing Complete ✅

| Step | Action | Reason |
|---|---|---|
| Remove price=0 | Dropped 874 rows | Invalid listings |
| log1p(price) | Created `log_price` column | Fix skew, align with RMSLE |
| Fill brand_name | Filled 632,682 nulls with "missing" | Preserve rows, signal no-brand |
| Fill category_name | Filled 6,327 nulls | Prevent split errors |
| Fill item_description | Filled 6 nulls | Prevent TF-IDF errors |
| Split category | Created cat1, cat2, cat3 | Model can use hierarchy |
| Clean text | Lowercase, remove [rm] | Better TF-IDF features |
| Save to .pkl | Saved to data/processed/ | Reusable for next notebook |